In [ ]:
from torch.utils.tensorboard import SummaryWriter
import time
import gymnasium as gym
import torch
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributions as D
import torch.optim as optim
import imageio
import matplotlib.pyplot as plt

In [2]:
env = gym.make("InvertedPendulum-v4")
state, _ = env.reset()
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
print(state_dim, action_dim)
print(state)

c:\Users\91930\anaconda3\envs\midas-py310-2\Lib\site-packages\gymnasium\envs\registration.py:512: DeprecationWarning: WARN: The environment InvertedPendulum-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


4 1
[ 0.00861074 -0.00192402 -0.00836523 -0.00861654]


In [3]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        self.mean = nn.Linear(128, action_dim)
        self.log_std = nn.Linear(128, action_dim)

    def forward(self, state):
        x = self.net(state)

        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, -20, 2)  # stability
        std = torch.exp(log_std)

        dist = D.Normal(mean, std)
        # print(f" dist is: {dist}")
        return dist

class Critic(nn.Module):
    def __init__(self, state_dim):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, state):
        return self.net(state)

In [4]:
actor = Actor(state_dim, action_dim)
critic = Critic(state_dim)

action_low = torch.tensor(env.action_space.low, dtype=torch.float32)
action_high = torch.tensor(env.action_space.high, dtype=torch.float32)

In [5]:
def select_action(state):
    state = torch.tensor(state, dtype=torch.float32)

    dist = actor(state)
    raw_action = dist.rsample()
    # print(raw_action)
    action = torch.tanh(raw_action)

    # scale to env bounds
    action_env = action_low + (action + 1) * 0.5 * (action_high - action_low)

    log_prob = dist.log_prob(raw_action).sum()
    entropy = dist.entropy().sum()

    return action_env.detach().numpy(), log_prob, entropy


In [6]:
action, log_prob, entropy = select_action(state)
print(action, log_prob, entropy)

[0.5580721] tensor(-1.0366, grad_fn=<SumBackward0>) tensor(1.5295, grad_fn=<SumBackward0>)


In [7]:
class Agent():
    def __init__(self, gamma=0.95, n_steps=3,
                 entropy_coef=0.01, entropy_decay=0.995,
                 env_name="Pendulum-v1"):

        self.env = gym.make(env_name)
        state_dim = self.env.observation_space.shape[0]
        action_dim = self.env.action_space.shape[0]

        self.actor = Actor(state_dim, action_dim)
        self.critic = Critic(state_dim)

        self.action_low = torch.tensor(self.env.action_space.low, dtype=torch.float32)
        self.action_high = torch.tensor(self.env.action_space.high, dtype=torch.float32)

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=1e-3)
        self.critic_opt = optim.Adam(self.critic.parameters(), lr=3e-3)

        self.gamma = gamma
        self.n_steps = n_steps
        self.entropy_coef = entropy_coef
        self.entropy_decay = entropy_decay

        # n-step buffers
        self.states = []
        self.rewards = []
        self.log_probs = []
        self.entropies = []

        # episode stats
        self.ep_reward = 0.0
        self.ep_length = 0

        # -------- logging history --------
        self.history_reward = []
        self.history_actor_loss = []
        self.history_critic_loss = []
        self.history_entropy = []
        self.history_value = []
        # ---------------------------------

    # --- training remains mostly unchanged ---
    def train(self, state, log_probs, entropy, reward, next_state, done):

        self.states.append(
            torch.tensor(state, dtype=torch.float32).unsqueeze(0).detach()
        )
        self.rewards.append(float(reward))
        self.log_probs.append(log_probs)
        self.entropies.append(entropy)

        if len(self.rewards) < self.n_steps and not done:
            return

        with torch.no_grad():
            next_state = torch.tensor(next_state, dtype=torch.float32).unsqueeze(0)
            R = 0.0 if done else self.critic(next_state).item()

        returns = []
        for r in reversed(self.rewards):
            R = r + self.gamma * R
            returns.insert(0, R)

        returns = torch.tensor(returns)

        state_tensor = torch.cat(self.states, dim=0)
        values = self.critic(state_tensor).squeeze()

        advantages = returns - values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        log_probs = torch.stack(self.log_probs)
        entropies = torch.stack(self.entropies)

        actor_loss = -(log_probs * advantages.detach()).mean() \
                     - self.entropy_coef * entropies.detach().mean()

        critic_loss = (returns.detach() - values).pow(2).mean()

        # -------- store logs --------
        self.history_actor_loss.append(actor_loss.item())
        self.history_critic_loss.append(critic_loss.item())
        self.history_value.append(values.mean().item())
        # ---------------------------

        self.actor_opt.zero_grad()
        actor_loss.backward()
        self.actor_opt.step()

        self.critic_opt.zero_grad()
        critic_loss.backward()
        self.critic_opt.step()

        if done:
            self.entropy_coef = max(0.001, self.entropy_coef * self.entropy_decay)

        self.states.clear()
        self.rewards.clear()
        self.log_probs.clear()
        self.entropies.clear()

    # --- stochastic action for training ---
    def select_action(self, state):
        state = torch.tensor(state, dtype=torch.float32).unsqueeze(0)

        dist = self.actor(state)
        raw_action = dist.rsample()
        action = torch.tanh(raw_action)

        action_env = self.action_low + (action + 1) * 0.5 * (
            self.action_high - self.action_low
        )

        log_prob = dist.log_prob(raw_action)
        log_prob -= torch.log(1 - torch.tanh(raw_action).pow(2) + 1e-6)
        log_prob = log_prob.sum()

        entropy = dist.entropy().sum()

        return action_env.squeeze(0).detach().numpy(), log_prob, entropy

    # --- deterministic action for evaluation (safe, no noise) ---
    def select_action_eval(self, state):
        state = torch.tensor(state, dtype=torch.float32).view(1, -1)  # <-- fixed shape

        with torch.no_grad():
            dist = self.actor(state)
            raw_action = dist.mean              # deterministic
            action = torch.tanh(raw_action)    # squash

            # scale to env range
            action_env = self.action_low + (action + 1) * 0.5 * (
                self.action_high - self.action_low
            )

        return action_env.squeeze(0).cpu().numpy()  # correct shape

    # --- evaluation, with optional RGB array rendering for GIFs ---
    def evaluate(self, num_episodes=10, render=False, save_frames=False):
        rewards = []
        all_frames = []

        for ep in range(num_episodes):
            state, _ = self.env.reset()
            done = False
            ep_reward = 0.0
            frames = []

            while not done:
                if render:
                    self.env.render()  # for human display

                action = self.select_action_eval(state)
                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                ep_reward += reward
                state = next_state

                if save_frames:
                    frame = self.env.render(mode='rgb_array')
                    frames.append(frame)

            rewards.append(ep_reward)
            if save_frames:
                all_frames.append(frames)

            print(f"[EVAL] Episode {ep:02d} | Reward: {ep_reward:.2f}")

        avg_reward = np.mean(rewards)
        std_reward = np.std(rewards)
        print(f"\n[EVAL SUMMARY] Avg Reward: {avg_reward:.2f} ± {std_reward:.2f}")

        if save_frames:
            return avg_reward, all_frames
        return avg_reward

    # --- training loop ---
    def run(self, number_of_episodes=200):
        for ep in range(number_of_episodes):
            state, _ = self.env.reset()
            done = False

            self.ep_reward = 0.0
            self.ep_length = 0

            while not done:
                action, log_prob, entropy = self.select_action(state)
                next_state, reward, terminated, truncated, _ = self.env.step(action)
                done = terminated or truncated

                self.ep_reward += reward
                self.ep_length += 1
                self.history_entropy.append(entropy.item())

                self.train(state, log_prob, entropy, reward, next_state, done)
                state = next_state

            self.history_reward.append(self.ep_reward)

            if ep % 10 == 0:
                print(
                    f"Episode {ep:03d} | "
                    f"Reward: {self.ep_reward:8.2f} | "
                    f"Len: {self.ep_length:4d} | "
                    f"EntropyCoef: {self.entropy_coef:.4f}"
                )
            if ep % 100 == 0 and ep > 0:
                self.plot_training()

    # --- plotting ---
    def plot_training(self):
        fig, axs = plt.subplots(2, 1, figsize=(6, 8))

        axs[0].plot(self.history_reward)
        axs[0].set_title("Episode Reward")
        axs[1].plot(self.history_value)
        axs[1].set_title("Mean Value Estimate V(s)")

        plt.tight_layout()
        plt.show()


In [ ]:
agent = Agent(gamma = 0.99, n_steps=10, entropy_coef=0.001, entropy_decay=0.995, env_name="Pendulum-v1")

In [17]:
agent.run(1)

Episode 000 | Reward: -1739.31 | Len:  200 | EntropyCoef: 0.0100


In [18]:
agent.evaluate(20, render=False)

[EVAL] Episode 00 | Reward: -1935.15
[EVAL] Episode 01 | Reward: -1287.73
[EVAL] Episode 02 | Reward: -1590.24
[EVAL] Episode 03 | Reward: -1511.00
[EVAL] Episode 04 | Reward: -1686.43
[EVAL] Episode 05 | Reward: -1831.52
[EVAL] Episode 06 | Reward: -1952.77
[EVAL] Episode 07 | Reward: -1413.81
[EVAL] Episode 08 | Reward: -986.97
[EVAL] Episode 09 | Reward: -1481.68
[EVAL] Episode 10 | Reward: -1907.36
[EVAL] Episode 11 | Reward: -1293.54
[EVAL] Episode 12 | Reward: -1258.59
[EVAL] Episode 13 | Reward: -1506.64
[EVAL] Episode 14 | Reward: -1315.16
[EVAL] Episode 15 | Reward: -1442.28
[EVAL] Episode 16 | Reward: -1760.96
[EVAL] Episode 17 | Reward: -1164.27
[EVAL] Episode 18 | Reward: -1499.84
[EVAL] Episode 19 | Reward: -1278.62

[EVAL SUMMARY] Avg Reward: -1505.23 ± 264.29


np.float64(-1505.2269644070668)

In [ ]:
render_env = gym.make("Pendulum-v1", render_mode="rgb_array")

frames = []
state, _ = render_env.reset()

done = False
total_reward = 0.0

while not done:
    state_tensor = torch.as_tensor(state, dtype=torch.float32).view(1, -1)

    with torch.no_grad():
        dist = agent.actor(state_tensor)
        action_tensor = torch.tanh(dist.mean) * 2.0

    action = action_tensor.squeeze(0).cpu().numpy()

    state, reward, terminated, truncated, _ = render_env.step(action)
    done = terminated or truncated
    total_reward += reward

    frame = render_env.render()
    frames.append(frame)

render_env.close()
print("Episode reward:", total_reward)




Episode reward: -1443.5307222603358


In [182]:
imageio.mimsave(
    "pendulum_eval.gif",
    frames,
    fps=30
)
print("Saved evaluation GIF. Total Reward:", total_reward)

Saved evaluation GIF. Total Reward: -1247.558113383126
